# Building prediction model dataset

## Config & imports

In [1]:
import os
os.chdir('..') 
print("Now working from:", os.getcwd())

Now working from: /Users/mikolajandrzejewski/Documents/GitHub/Bachelor


In [ ]:
import sys
sys.path.append('embeddings')

import os
import ast
import glob
import pickle
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from lorentz import Lorentz
from collections import defaultdict


In [ ]:
MGD_ROOT = 'datasets'  

CHARTS_DIR = os.path.join(MGD_ROOT, 'Charts')
ARTISTS_CSV = os.path.join(MGD_ROOT, 'Artists', 'spotify_artists_info_complete.csv')
GENRE_NET_DIR = os.path.join(MGD_ROOT, 'Original')

CKPT = 'ckpt/final_enao_graph.ckpt'
VOCAB = 'enao_vocab.pkl' 
DIM = 2

MARKETS = ['au', 'br', 'ca', 'de', 'fr', 'gb', 'global', 'jp', 'us']
YEARS = [2017, 2018, 2019]

OUTPUT_POPULARITY = 'prediction_model/genre_popularity.csv'
OUTPUT_RANKING = 'prediction_model/genre_ranking.csv'
OUTPUT_DATASET = 'prediction_model/model_dataset.csv'

## 1. Loading embeddings

In [4]:
genres_list = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres_list)
genre_to_idx = {g: i for i, g in enumerate(genres_list)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

lorentz_table = net.table.weight.data.cpu().numpy()

def lorentz_distance(u, v):
    inner = -u[0]*v[0] + np.dot(u[1:], v[1:])
    inner = np.clip(-inner, 1 + 1e-6, None)
    return float(np.arccosh(inner))

def get_embedding(genre_name):
    if genre_name not in genre_to_idx:
        return None
    return lorentz_table[genre_to_idx[genre_name] + 1]

print(f'Loaded {n_items} genre embeddings')

Loaded 6280 genre embeddings


## 2. Building genre popularity per market per year

For each (market, year) we sum the weekly streams of every chart song, attributed to each genre of that song's artist.


genre_popularity(genre, market, year) =
    sum of streams across all weeks
    for all songs whose artist has that genre


In [5]:
artists_df = pd.read_csv(ARTISTS_CSV, sep='\t')

def parse_genres(s):
    try:
        return ast.literal_eval(s)
    except:
        return []

artists_df['genres_parsed'] = artists_df['genres'].apply(parse_genres)
artist_genres = dict(zip(artists_df['name'].str.strip(),
                         artists_df['genres_parsed']))

print(f'Artists loaded: {len(artist_genres)}')
print('Sample:', list(artist_genres.items())[:3])

Artists loaded: 3584
Sample: [('AURORA', ['norwegian pop']), ('MHD', ['francoton', 'french hip hop', 'pop urbaine']), ('Chuck Berry', ['blues rock', 'classic rock', 'folk rock', 'rock', 'rock-and-roll', 'rockabilly', 'roots rock', 'soul'])]


In [6]:
popularity_rows = []
skipped_artists = set()

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        weekly_files = glob.glob(os.path.join(charts_path, '*.csv'))
        if not weekly_files:
            continue

        genre_streams = defaultdict(float)

        for fpath in weekly_files:
            try:
                chart = pd.read_csv(fpath, sep='\t')
            except Exception:
                continue

            for _, row in chart.iterrows():
                artist_name = str(row['artist']).strip()
                streams = row['streams']
                genres = artist_genres.get(artist_name, [])

                if not genres:
                    skipped_artists.add(artist_name)
                    continue

                for genre in genres:
                    genre_streams[genre] += streams

        for genre, total_streams in genre_streams.items():
            popularity_rows.append({
                'market':       market,
                'year':         year,
                'genre':        genre,
                'total_streams': total_streams
            })


popularity_df = pd.DataFrame(popularity_rows)
popularity_df.to_csv(OUTPUT_POPULARITY, index=False)

print(f'\nGenre popularity table: {len(popularity_df)} rows')
print(f'Artists with no genres (skipped): {len(skipped_artists)}')


Genre popularity table: 6258 rows
Artists with no genres (skipped): 131


## 2b. Computing ranking-based popularity

In addition to stream-based popularity, we compute ranking-based metrics:
- avg_ranking: Average chart position (lower = better)
- ranking_score: Weighted score (rank 1 = 200 pts, rank 200 = 1 pt; higher = better)
- n_chart_appearances: Number of weeks on charts

In [7]:
ranking_rows = []

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        genre_rankings = defaultdict(list)
        
        week_files = sorted(glob.glob(os.path.join(charts_path, '*.csv')))
        skipped_files = 0
        for week_file in week_files:
            try:
                week_df = pd.read_csv(week_file, sep='\t')
                
                if 'position' not in week_df.columns or 'artist' not in week_df.columns:
                    skipped_files += 1
                    continue
                    
            except Exception as e:
                skipped_files += 1
                continue
            
            for _, row in week_df.iterrows():
                try:
                    artist_name = str(row['artist']).strip()
                    rank = int(row['position'])
                except:
                    continue
                
                if artist_name not in artist_genres:
                    continue
                    
                for genre in artist_genres[artist_name]:
                    genre_rankings[genre].append(rank)
        
        if skipped_files > 0:
            print(f'  {market} {year}: skipped {skipped_files} files')
        
        for genre, ranks in genre_rankings.items():
            avg_rank = np.mean(ranks)
            weighted_score = sum(200 - r + 1 for r in ranks)
            
            ranking_rows.append({
                'market': market,
                'year': year,
                'genre': genre,
                'avg_ranking': avg_rank,
                'ranking_score': weighted_score,
                'n_chart_appearances': len(ranks)
            })
    

ranking_df = pd.DataFrame(ranking_rows)
ranking_df.to_csv(OUTPUT_RANKING, index=False)
print(f'\nSaved to: {OUTPUT_RANKING}')


Saved to: prediction_model/genre_ranking.csv


## 3. Building the enriched model dataset

For each (market, year) genre network file we:
- Remove self-loops
- Compute hyperbolic distance for each pair
- Attach popularity_source and popularity_target from the popularity table
- Add market and year columns

Then stack all markets and years into one CSV.

In [8]:
pop_lookup = {}
for _, row in popularity_df.iterrows():
    pop_lookup[(row['market'], row['year'], row['genre'])] = row['total_streams']

rank_lookup = {}
rank_score_lookup = {}
for _, row in ranking_df.iterrows():
    rank_lookup[(row['market'], row['year'], row['genre'])] = row['avg_ranking']
    rank_score_lookup[(row['market'], row['year'], row['genre'])] = row['ranking_score']

all_rows = []
skipped_no_embedding = 0
skipped_no_popularity = 0

for market in MARKETS:
    for year in YEARS:
        net_path = os.path.join(GENRE_NET_DIR, market,
                                f'{market}-genre_network-{year}.csv')
        if not os.path.exists(net_path):
            print(f'  Missing genre network: {net_path}')
            continue

        gn = pd.read_csv(net_path, sep='\t')
        gn = gn[gn['source'] != gn['target']] 

        for _, row in gn.iterrows():
            src, tgt = row['source'], row['target']

            # hyperbolic distance
            u = get_embedding(src)
            v = get_embedding(tgt)
            if u is None or v is None:
                skipped_no_embedding += 1
                continue

            dist = lorentz_distance(u, v)

            # genre popularity (streams)
            pop_src = pop_lookup.get((market, year, src))
            pop_tgt = pop_lookup.get((market, year, tgt))

            if pop_src is None or pop_tgt is None:
                skipped_no_popularity += 1
                continue

            # Genre ranking
            rank_src = rank_lookup.get((market, year, src))
            rank_tgt = rank_lookup.get((market, year, tgt))
            rank_score_src = rank_score_lookup.get((market, year, src))
            rank_score_tgt = rank_score_lookup.get((market, year, tgt))

            all_rows.append({
                'market':       market,
                'year':         year,
                'source':       src,
                'target':       tgt,
                'weight':       row['weight'],
                'avg_streams':  row['avg_streams'],
                'distance':     dist,
                
                # Streams-based popularity
                'popularity_source':   pop_src,
                'popularity_target':   pop_tgt,
                
                # Ranking-based popularity
                'ranking_source':      rank_src,
                'ranking_target':      rank_tgt,
                'ranking_score_source': rank_score_src,
                'ranking_score_target': rank_score_tgt,
                
                # Log transforms (streams)
                'log_streams':  np.log10(row['avg_streams'] + 1),
                'log_popularity_source':  np.log10(pop_src + 1),
                'log_popularity_target':  np.log10(pop_tgt + 1),
                'log_weight':   np.log10(row['weight'] + 1),
                
                # Log transforms (ranking)
                'log_ranking_score_source': np.log10(rank_score_src + 1) if rank_score_src else None,
                'log_ranking_score_target': np.log10(rank_score_tgt + 1) if rank_score_tgt else None,
            })

model_df = pd.DataFrame(all_rows)

model_df['distance_norm'] = 0.0
for market in model_df['market'].unique():
    mask = model_df['market'] == market
    scaler = MinMaxScaler()
    model_df.loc[mask, 'distance_norm'] = scaler.fit_transform(
        model_df.loc[mask, ['distance']]
    ).flatten()

scaler_global = MinMaxScaler()
model_df['distance_norm_global'] = scaler_global.fit_transform(
    model_df[['distance']]
).flatten()

model_df.to_csv(OUTPUT_DATASET, index=False)

print(f'Model dataset: {len(model_df)} rows')
print(f'Skipped (no embedding): {skipped_no_embedding}')
print(f'Skipped (no popularity): {skipped_no_popularity}')

Model dataset: 43661 rows
Skipped (no embedding): 4265
Skipped (no popularity): 13067


## 4. Quick sanity check

In [9]:
print('=== Sample rows ===')
print(model_df[['market', 'year', 'source', 'target',
                'distance', 'log_popularity_source', 'log_popularity_target',
                'log_ranking_score_source', 'log_ranking_score_target',
                'log_weight', 'log_streams']]
      .head(10).to_string(index=False))
print()
print('=== Missing values ===')
print(model_df.isnull().sum())

=== Sample rows ===
market  year    source        target  distance  log_popularity_source  log_popularity_target  log_ranking_score_source  log_ranking_score_target  log_weight  log_streams
    au  2017 dance pop           pop  3.045446               9.104764               9.320063                  5.597187                  5.814359    2.399674     6.769483
    au  2017       pop       pop rap  6.244612               9.320063               8.802308                  5.814359                  5.303827    2.340444     6.664606
    au  2017       pop           rap  6.182096               9.320063               8.830240                  5.814359                  5.321952    2.305351     6.541656
    au  2017   pop rap           rap  0.151346               8.802308               8.830240                  5.303827                  5.321952    2.283301     6.494957
    au  2017 dance pop       pop rap  4.312433               9.104764               8.802308                  5.597187            